
# Train Qwen3 Embeddings for French and Spanish (Separate Runs)

This notebook fine-tunes `Qwen/Qwen3-Embedding-8B` with LoRA using contrastive learning.
It trains **two separate adapters**: one for French (`fr`) and one for Spanish (`es`).

Use this when you want strongest multilingual base performance plus language-specific specialization.



## 1) Environment

Run this on a GPU machine (A100/H100 preferred). 8B training requires substantial VRAM.

If needed, install dependencies in this kernel:
```bash
pip install -U torch transformers datasets peft accelerate scikit-learn pandas numpy tqdm sentencepiece
```


In [ ]:

import os
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType


In [ ]:

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)



## 2) Configuration

Expected training file schema (CSV/Parquet):
- `language`: `fr` or `es`
- `anchor`: source/query text
- `positive`: matched positive text
- optional `hard_negative`: challenging non-match

You can use one combined file with both languages, or separate files.


In [ ]:

@dataclass
class TrainConfig:
    base_model: str = "Qwen/Qwen3-Embedding-8B"

    # Data
    train_path: str = "data/processed/embedding_pairs.csv"  # change to your file
    file_type: str = "csv"  # csv or parquet

    # Optimization
    batch_size: int = 16
    epochs: int = 2
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    temperature: float = 0.05
    max_length: int = 256

    # LoRA
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # Output
    output_root: str = "artifacts/embeddings"

cfg = TrainConfig()
cfg



## 3) Data Loading


In [ ]:

def load_pairs(path: str, file_type: str = "csv") -> pd.DataFrame:
    if file_type == "csv":
        df = pd.read_csv(path)
    elif file_type == "parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError("file_type must be 'csv' or 'parquet'")

    required = {"language", "anchor", "positive"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "hard_negative" not in df.columns:
        df["hard_negative"] = ""

    df = df.dropna(subset=["language", "anchor", "positive"]).copy()
    df["language"] = df["language"].str.lower().str.strip()
    df = df[df["language"].isin(["fr", "es"])].reset_index(drop=True)
    return df


df_all = load_pairs(cfg.train_path, cfg.file_type)
print(df_all["language"].value_counts())
df_all.head(3)



## 4) Modeling Utilities


In [ ]:

def make_model_and_tokenizer(base_model: str):
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

    # Use bf16 on modern GPUs; fallback to fp16 where bf16 unavailable.
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModel.from_pretrained(
        base_model,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map=None,
    )

    peft_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        inference_mode=False,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
    )

    model = get_peft_model(model, peft_cfg)
    model.to(device)
    model.print_trainable_parameters()
    return model, tokenizer


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return pooled


def encode_texts(model, tokenizer, texts, max_length=256):
    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)

    out = model(**batch)
    if hasattr(out, "last_hidden_state"):
        emb = mean_pool(out.last_hidden_state, batch["attention_mask"])
    else:
        emb = out[0].mean(dim=1)

    emb = F.normalize(emb, p=2, dim=1)
    return emb


In [ ]:

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame):
        self.anchor = df["anchor"].tolist()
        self.positive = df["positive"].tolist()
        self.hard_negative = df["hard_negative"].fillna("").tolist()

    def __len__(self):
        return len(self.anchor)

    def __getitem__(self, idx):
        return {
            "anchor": self.anchor[idx],
            "positive": self.positive[idx],
            "hard_negative": self.hard_negative[idx],
        }


def collate_fn(items):
    return {
        "anchor": [x["anchor"] for x in items],
        "positive": [x["positive"] for x in items],
        "hard_negative": [x["hard_negative"] for x in items],
    }



## 5) Training and Evaluation


In [ ]:

@torch.no_grad()
def evaluate_recall_at_k(model, tokenizer, df_eval, k_values=(1, 5, 10)):
    model.eval()

    anchors = df_eval["anchor"].tolist()
    positives = df_eval["positive"].tolist()

    emb_a = []
    emb_p = []
    bs = cfg.batch_size

    for i in range(0, len(anchors), bs):
        emb_a.append(encode_texts(model, tokenizer, anchors[i:i+bs], cfg.max_length).cpu())
        emb_p.append(encode_texts(model, tokenizer, positives[i:i+bs], cfg.max_length).cpu())

    emb_a = torch.cat(emb_a, dim=0)
    emb_p = torch.cat(emb_p, dim=0)
    sim = emb_a @ emb_p.T

    ranks = torch.argsort(sim, dim=1, descending=True)
    labels = torch.arange(sim.size(0)).unsqueeze(1)

    metrics = {}
    for k in k_values:
        topk = ranks[:, :k]
        hit = (topk == labels).any(dim=1).float().mean().item()
        metrics[f"recall@{k}"] = hit

    return metrics


def train_one_language(lang: str, cfg: TrainConfig):
    assert lang in {"fr", "es"}

    df_lang = df_all[df_all["language"] == lang].reset_index(drop=True)
    if len(df_lang) < 100:
        raise ValueError(f"Not enough samples for {lang}. Need at least 100 rows, got {len(df_lang)}")

    train_df, val_df = train_test_split(df_lang, test_size=0.1, random_state=SEED)

    train_ds = PairDataset(train_df)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        drop_last=True,
    )

    model, tokenizer = make_model_and_tokenizer(cfg.base_model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_r1 = -1.0
    out_dir = os.path.join(cfg.output_root, f"qwen3-emb-8b-{lang}-lora")
    os.makedirs(out_dir, exist_ok=True)

    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f"{lang} epoch {epoch+1}/{cfg.epochs}")
        for batch in pbar:
            emb_a = encode_texts(model, tokenizer, batch["anchor"], cfg.max_length)
            emb_p = encode_texts(model, tokenizer, batch["positive"], cfg.max_length)

            sim_ap = (emb_a @ emb_p.T) / cfg.temperature
            labels = torch.arange(sim_ap.size(0), device=sim_ap.device)

            hard_neg = [x for x in batch["hard_negative"] if isinstance(x, str) and x.strip()]
            if len(hard_neg) == len(batch["anchor"]):
                emb_n = encode_texts(model, tokenizer, batch["hard_negative"], cfg.max_length)
                sim_an = (emb_a @ emb_n.T) / cfg.temperature
                logits = torch.cat([sim_ap, sim_an], dim=1)
            else:
                logits = sim_ap

            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = running_loss / max(1, len(train_loader))
        metrics = evaluate_recall_at_k(model, tokenizer, val_df)
        r1 = metrics["recall@1"]

        print(f"[{lang}] epoch={epoch+1} avg_loss={avg_loss:.4f} metrics={metrics}")

        if r1 > best_r1:
            best_r1 = r1
            model.save_pretrained(out_dir)
            tokenizer.save_pretrained(out_dir)
            print(f"[{lang}] saved best adapter to: {out_dir}")

    return out_dir, best_r1



## 6) Run French and Spanish Training (Separate)

This cell launches two independent runs and writes adapters under:
- `artifacts/embeddings/qwen3-emb-8b-fr-lora`
- `artifacts/embeddings/qwen3-emb-8b-es-lora`


In [ ]:

fr_dir, fr_best = train_one_language("fr", cfg)
es_dir, es_best = train_one_language("es", cfg)

print("French adapter:", fr_dir, "best recall@1:", fr_best)
print("Spanish adapter:", es_dir, "best recall@1:", es_best)



## 7) Encode Texts with a Trained Adapter


In [ ]:

from peft import PeftModel


def load_adapter_for_inference(adapter_dir: str, base_model: str = cfg.base_model):
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base = AutoModel.from_pretrained(
        base_model,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map=None,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.to(device)
    model.eval()
    return model, tokenizer


# Example:
# model_fr, tok_fr = load_adapter_for_inference("artifacts/embeddings/qwen3-emb-8b-fr-lora")
# vecs = encode_texts(model_fr, tok_fr, ["bonjour", "salut"], max_length=cfg.max_length)
# print(vecs.shape)



## Notes

- For maximum quality, increase data quality, use hard negatives, and run more epochs.
- If you have multi-GPU infrastructure, use `accelerate`/DDP or DeepSpeed for faster runs.
- If your input text includes task instructions (query/document), keep formatting consistent in training and inference.


## 8) Build and Use FAISS Index

This section encodes a retrieval corpus with your trained adapter and builds a FAISS index.


In [ ]:

import faiss


def load_corpus(path: str, file_type: str = "csv", text_col: str = "text") -> pd.DataFrame:
    if file_type == "csv":
        df = pd.read_csv(path)
    elif file_type == "parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError("file_type must be 'csv' or 'parquet'")

    if text_col not in df.columns:
        raise ValueError(f"Missing text column: {text_col}")

    df = df.dropna(subset=[text_col]).reset_index(drop=True)
    return df


def encode_corpus(model, tokenizer, texts, batch_size=128, max_length=256):
    all_emb = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding corpus"):
            batch = texts[i:i+batch_size]
            emb = encode_texts(model, tokenizer, batch, max_length=max_length)
            all_emb.append(emb.detach().cpu().numpy())
    mat = np.vstack(all_emb).astype("float32")
    return mat


def build_faiss_ip_index(embedding_matrix: np.ndarray):
    embs = embedding_matrix.copy()
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, embs


def save_faiss_artifacts(index, embeddings, out_dir: str, prefix: str):
    os.makedirs(out_dir, exist_ok=True)
    idx_path = os.path.join(out_dir, f"{prefix}.faiss")
    emb_path = os.path.join(out_dir, f"{prefix}_embeddings.npy")
    faiss.write_index(index, idx_path)
    np.save(emb_path, embeddings)
    return idx_path, emb_path


def faiss_search(index, query_embeddings: np.ndarray, top_k=10):
    q = query_embeddings.copy().astype("float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, top_k)
    return scores, ids


In [ ]:

# Configure corpus/index build
INDEX_LANG = "fr"  # "fr" or "es"
CORPUS_PATH = "data/processed/retrieval_corpus.csv"  # change me
CORPUS_FILE_TYPE = "csv"
CORPUS_TEXT_COL = "text"
INDEX_OUT_DIR = "artifacts/embeddings/faiss"

# Pick adapter
adapter_dir = f"artifacts/embeddings/qwen3-emb-8b-{INDEX_LANG}-lora"
model_idx, tok_idx = load_adapter_for_inference(adapter_dir)

# Load and encode corpus
corpus_df = load_corpus(CORPUS_PATH, CORPUS_FILE_TYPE, CORPUS_TEXT_COL)
corpus_texts = corpus_df[CORPUS_TEXT_COL].astype(str).tolist()
corpus_embs = encode_corpus(model_idx, tok_idx, corpus_texts, batch_size=128, max_length=cfg.max_length)

# Build and save FAISS index
index, norm_embs = build_faiss_ip_index(corpus_embs)
idx_path, emb_path = save_faiss_artifacts(index, norm_embs, INDEX_OUT_DIR, prefix=f"{INDEX_LANG}_qwen3_emb8b")

print("index saved:", idx_path)
print("embeddings saved:", emb_path)
print("corpus size:", len(corpus_df), "dim:", norm_embs.shape[1])


In [ ]:

# Example query search
queries = [
    "Votre texte de requete ici" if INDEX_LANG == "fr" else "Tu texto de consulta aqui"
]

q_emb = encode_corpus(model_idx, tok_idx, queries, batch_size=32, max_length=cfg.max_length)
scores, ids = faiss_search(index, q_emb, top_k=5)

for qi, q in enumerate(queries):
    print("\nQUERY:", q)
    for rank, (doc_id, score) in enumerate(zip(ids[qi], scores[qi]), start=1):
        row = corpus_df.iloc[int(doc_id)]
        print(f"{rank}. id={doc_id} score={score:.4f} text={row[CORPUS_TEXT_COL]}")


## 9) Optional Retrieval Evaluation

Expected query file columns: `query`, `positive_doc_id` (matching corpus row indices).


In [ ]:

@torch.no_grad()
def evaluate_faiss_recall(index, model, tokenizer, query_df, k_values=(1, 5, 10)):
    queries = query_df["query"].astype(str).tolist()
    y_true = query_df["positive_doc_id"].astype(int).tolist()

    q_emb = encode_corpus(model, tokenizer, queries, batch_size=128, max_length=cfg.max_length)
    max_k = max(k_values)
    _, ids = faiss_search(index, q_emb, top_k=max_k)

    metrics = {}
    for k in k_values:
        hit = 0
        for i, true_id in enumerate(y_true):
            if int(true_id) in set(ids[i, :k].tolist()):
                hit += 1
        metrics[f"recall@{k}"] = hit / len(y_true)
    return metrics


# Example:
# qdf = pd.read_csv("data/processed/retrieval_queries_fr.csv")
# metrics = evaluate_faiss_recall(index, model_idx, tok_idx, qdf, k_values=(1,5,10))
# print(metrics)
